# 03 - Silver: Tipagem, derivacao e classificacao de negocio

## Objetivo

Transformar os dados da Silver Staging em uma camada analítica estruturada.

As operacoes realizadas nesta etapa sao:

- tipagem explicita de datas, numeros e textos;
- separacao de codigo e descricao de municipio;
- criacao de colunas derivadas de negocio;
- classificacao de CID por capitulo e grupo;
- classificacao de especie por tipo de beneficio;
- sinalizacao de CID informado ou nao informado.

A tabela gerada e:

`afastamento_inss.silver.beneficios_concedidos`

In [0]:
# ============================================================
# IMPORTACOES
# ============================================================

from pyspark.sql.functions import (
    col,
    count,
    when,
    trim,
    lit,
    current_timestamp,
    try_to_date,
    coalesce,
    regexp_replace,
    regexp_extract,
    substring,
    split,
    upper,
    expr,
    date_format
)
from pyspark.sql.types import (
    IntegerType,
    DoubleType
)


In [0]:
# ============================================================
# PARAMETROS
# ============================================================

TABELA_ORIGEM  = "afastamento_inss.silver.stg_beneficios_concedidos"
TABELA_DESTINO = "afastamento_inss.silver.beneficios_concedidos"

print(f"Origem : {TABELA_ORIGEM}")
print(f"Destino: {TABELA_DESTINO}")


## 1. Leitura da Silver Staging

A Silver sempre le da Staging e nunca da Bronze diretamente.

Isso garante que os tratamentos de qualidade ja foram aplicados
antes de qualquer transformacao de negocio.

In [0]:
df_staging = spark.table(TABELA_ORIGEM)

print(f"Linhas : {df_staging.count()}")
print(f"Colunas: {len(df_staging.columns)}")


## 2. Tipagem de datas

As datas no arquivo original estao no formato DD/MM/YYYY como string.

Excecao: dt_dcb contem o valor 00/00/0000 para beneficios sem
data de cessacao. Esse valor nao pode ser convertido para DateType
e ja foi mapeado para null na Silver Staging.

As colunas convertidas sao:
- dt_nascimento
- dt_dcb
- dt_ddb
- dt_dib

In [0]:
df_silver = df_staging

# Converter datas para DateType.
# Os CSVs trazem as datas como "yyyy-MM-dd HH:mm:ss" (ex.: 1972-08-15 00:00:00);
# "dd/MM/yyyy" e mantido como alternativa. "00/00/0000" (sem data) vira null.
COLUNAS_DATA = [
    "dt_nascimento",
    "dt_dcb",
    "dt_ddb",
    "dt_dib"
]

for c in COLUNAS_DATA:
    df_silver = df_silver.withColumn(
        c,
        coalesce(
            try_to_date(col(c), "yyyy-MM-dd HH:mm:ss"),
            try_to_date(col(c), "dd/MM/yyyy")
        )
    )

print("Tipagem de datas aplicada.")
print(f"Colunas convertidas: {COLUNAS_DATA}")

# Verificar quantos nulls gerados por datas invalidas
for c in COLUNAS_DATA:
    nulos = df_silver.filter(col(c).isNull()).count()
    print(f"  {c:<25} nulls: {nulos:>8,}")


In [0]:
# ============================================================
# NORMALIZACAO DE competencia_concessao
# ============================================================
# A coluna competencia_concessao chega da staging com formatos
# mistos: a maioria no padrao yyyyMM (ex: 202306), mas algumas
# linhas trazem timestamp completo (ex: 2024-06-01 00:00:00) ou
# texto que nao representa uma competencia (ex: "Amp. Social...").
# Aqui normalizamos para o padrao yyyyMM string, preservando NULL.

df_silver = df_silver.withColumn(
    "competencia_concessao",
    when(
        col("competencia_concessao").rlike(r"^\d{6}$"),
        col("competencia_concessao")
    ).otherwise(
        when(
            col("competencia_concessao").isNotNull(),
            date_format(try_to_date(col("competencia_concessao")), "yyyyMM")
        )
    )
)

competencias_invalidas = df_silver.filter(
    col("competencia_concessao").isNotNull()
    & ~col("competencia_concessao").rlike(r"^\d{6}$")
).count()

print("Normalizacao de competencia_concessao aplicada.")
print(f"Competencias com formato invalido apos normalizacao: {competencias_invalidas}")


## 3. Tipagem de campos numericos

O campo qt_sm_rmi usa virgula como separador decimal.
Antes de converter para DoubleType e necessario substituir
a virgula por ponto.

O campo qt_anos_contribuicao e um inteiro.
O valor 0 e valido e representa zero anos de contribuicao.

In [0]:
# qt_sm_rmi: virgula -> ponto -> double
df_silver = df_silver.withColumn(
    "qt_sm_rmi",
    expr("try_cast(regexp_replace(qt_sm_rmi, ',', '.') as double)")
)

# qt_anos_contribuicao: string -> integer
df_silver = df_silver.withColumn(
    "qt_anos_contribuicao",
    col("qt_anos_contribuicao").cast(IntegerType())
)

print("Tipagem numerica aplicada.")


## Normalização de códigos (zeros à esquerda)

Os mesmos códigos aparecem com e sem zeros à esquerda conforme o arquivo (ex.: `aps_cod` `2001050` e `02001050`; `especie_cod` `04` e `4`; `despacho_cod` `00` e `0`). Os zeros são removidos para que cada código tenha uma única representação. Também são colapsados os espaços duplicados em `aps_desc`.

In [0]:
def sem_zeros_esquerda(nome):
    """Remove zeros a esquerda de codigos puramente numericos ('00' -> '0', '04' -> '4')."""
    return when(
        col(nome).rlike(r"^\d+$"),
        regexp_replace(col(nome), r"^0+(?=\d)", "")
    ).otherwise(col(nome))

for c in ["aps_cod", "especie_cod", "despacho_cod"]:
    df_silver = df_silver.withColumn(c, sem_zeros_esquerda(c))

df_silver = df_silver.withColumn("aps_desc", regexp_replace(trim(col("aps_desc")), r"\s+", " "))

print("Codigos normalizados: aps_cod, especie_cod, despacho_cod")


## 4. Separacao de codigo e nome do municipio

O campo mun_resid armazena codigo e nome do municipio
no mesmo valor, separados por hifen.

Exemplo: 21504-SP-Sao Paulo

A separacao gera duas colunas:
- mun_cod  : codigo IBGE do municipio
- mun_nome : nome do municipio com UF

In [0]:
df_silver = df_silver.withColumn(
    "mun_cod",
    when(
        col("mun_resid").isNotNull(),
        split(col("mun_resid"), "-").getItem(0)
    ).otherwise(None)
)

df_silver = df_silver.withColumn(
    "mun_nome",
    when(
        col("mun_resid").isNotNull(),
        trim(
            regexp_replace(col("mun_resid"), r"^\d+-", "")
        )
    ).otherwise(None)
)

print("Separacao de municipio aplicada.")
display(
    df_silver
    .select("mun_resid", "mun_cod", "mun_nome")
    .filter(col("mun_resid").isNotNull())
    .limit(10)
)


## Normalização do CID

Alguns `cid_cod` chegam da fonte com formatação inconsistente (ex.: `M-54`, `H54.`, `00F840`). O código é normalizado para o padrão `LNN` ou `LNNN` (ex.: `M54`, `F840`). Códigos que continuam inválidos após a limpeza (ex.: `N`, `200`, `M5`) viram `null` e são contados em `cid_cod_invalido` para auditoria.

In [0]:
cid_limpo = regexp_replace(upper(trim(col("cid_cod"))), r"[^A-Z0-9]", "")
cid_norm  = regexp_extract(cid_limpo, r"^0*([A-Z]\d{2}\d?)$", 1)

df_silver = (
    df_silver
    .withColumn("_cid_norm", cid_norm)
    .withColumn(
        "cid_cod_invalido",
        when(col("cid_cod").isNotNull() & (col("_cid_norm") == ""), lit(1)).otherwise(lit(0))
    )
    .withColumn(
        "cid_cod",
        when(col("cid_cod").isNull(), None)
        .when(col("_cid_norm") == "", None)
        .otherwise(col("_cid_norm"))
    )
    .drop("_cid_norm")
)

print("Normalizacao de cid_cod aplicada.")
print(f"Codigos CID invalidos (anulados): {df_silver.filter(col('cid_cod_invalido') == 1).count():,}")


## 5. Classificacao de CID

O campo cid_cod contem o codigo CID-10 do diagnostico.

Sao derivadas duas colunas:

cid_capitulo:
- Letra inicial do codigo CID
- Identifica o capitulo da CID-10
- Exemplos: F (mental), M (osteomuscular), K (digestivo)

cid_grupo:
- Classificacao relevante para o projeto
- mental        : capitulo F (transtornos mentais)
- osteomuscular : capitulo M (sistema osteomuscular)
- cardiovascular : capitulo I (aparelho circulatorio)
- respiratorio  : capitulo J (aparelho respiratorio)
- outros        : demais capitulos com CID informado
- nao_informado : CID ausente

O cid_capitulo e normalizado para maiuscula para corrigir
codigos CID registrados com letra minuscula na fonte.

cid_status:
- informado     : cid_cod preenchido
- nao_informado : cid_cod null

In [0]:
# cid_capitulo: primeira letra do cid_cod, normalizada para maiuscula
df_silver = df_silver.withColumn(
    "cid_capitulo",
    when(
        col("cid_cod").isNotNull(),
        upper(substring(col("cid_cod"), 1, 1))
    ).otherwise(None)
)

# cid_grupo: classificacao de negocio expandida com todos os capitulos CID-10
# cid_categoria: 3 primeiros caracteres (ex.: F32 para F320); cid_num: numero da categoria
df_silver = (
    df_silver
    .withColumn("cid_categoria", when(col("cid_cod").isNotNull(), substring(col("cid_cod"), 1, 3)))
    .withColumn("cid_num", substring(col("cid_cod"), 2, 2).cast("int"))
)

df_silver = df_silver.withColumn(
    "cid_grupo",
    when(col("cid_cod").isNull(),             "nao_informado")
    .when(col("cid_capitulo") == "F",         "mental")
    .when(col("cid_capitulo") == "M",         "osteomuscular")
    .when(col("cid_capitulo") == "I",         "cardiovascular")
    .when(col("cid_capitulo") == "J",         "respiratorio")
    .when(col("cid_capitulo") == "S",         "lesoes_causas_externas")
    .when(col("cid_capitulo") == "T",         "lesoes_causas_externas")
    .when(col("cid_capitulo") == "K",         "digestivo")
    .when(col("cid_capitulo") == "N",         "geniturinario")
    .when(col("cid_capitulo") == "C",         "neoplasias")
    # D00-D48 neoplasias; D50-D89 sangue e sistema imunitario
    .when((col("cid_capitulo") == "D") & (col("cid_num") <= 48), "neoplasias")
    .when(col("cid_capitulo") == "D",         "sangue_imunitario")
    .when(col("cid_capitulo") == "G",         "nervoso")
    .when(col("cid_capitulo") == "E",         "endocrino_metabolico")
    .when(col("cid_capitulo") == "A",         "infecciosas")
    .when(col("cid_capitulo") == "B",         "infecciosas")
    .when(col("cid_capitulo") == "L",         "pele")
    .when(col("cid_capitulo") == "H",         "sentidos")
    .when(col("cid_capitulo") == "O",         "gravidez_parto")
    .when(col("cid_capitulo") == "Q",         "congenitas")
    .when(col("cid_capitulo") == "P",         "perinatal")
    .when(col("cid_capitulo") == "V",         "causas_externas")
    .when(col("cid_capitulo") == "W",         "causas_externas")
    .when(col("cid_capitulo") == "X",         "causas_externas")
    .when(col("cid_capitulo") == "Y",         "causas_externas")
    .when(col("cid_capitulo") == "Z",         "fatores_saude")
    .when(col("cid_capitulo") == "R",         "sintomas")
    .when(col("cid_capitulo") == "U",         "especiais")
    .otherwise(                               "outros")
)

# cid_subgrupo: detalhamento dos grupos mental (F) e osteomuscular (M)
n = col("cid_num")
cat = col("cid_categoria")
df_silver = df_silver.withColumn(
    "cid_subgrupo",
    when(col("cid_cod").isNull(), None)
    .when(col("cid_capitulo") == "F",
        when(n.between(0, 9),   "transtornos_organicos")
        .when(n.between(10, 19), "uso_substancias")
        .when(n.between(20, 29), "esquizofrenia_psicoses")
        .when(cat == "F31",      "bipolar")
        .when(cat.isin("F32", "F33"), "depressao")
        .when(n.between(30, 39), "outros_humor")
        .when(cat.isin("F40", "F41"), "ansiedade")
        .when(cat == "F43",      "estresse_adaptacao")
        .when(n.between(40, 48), "outros_neuroticos")
        .otherwise("outros_mentais"))
    .when(col("cid_capitulo") == "M",
        when(n.between(40, 54), "dorsopatias")
        .otherwise("outros_osteomusculares"))
    .otherwise("nao_aplicavel")
).drop("cid_num")

# cid_status
df_silver = df_silver.withColumn(
    "cid_status",
    when(col("cid_cod").isNotNull(), "informado")
    .otherwise("nao_informado")
)

print("Classificacao de CID aplicada.")
display(
    df_silver
    .groupBy("cid_grupo")
    .count()
    .orderBy("count", ascending=False)
)


## 6. Classificacao de especie por tipo de beneficio

O campo especie_cod identifica o tipo de beneficio concedido.

A coluna derivada tipo_beneficio agrupa as especies em categorias
relevantes para a analise de afastamentos:

- afastamento         : auxilio-doenca e auxilio-acidente
- aposentadoria       : aposentadorias por idade, tempo, invalidez (inclui acidentária)
- pensao              : pensao por morte (previdenciária, estatutária, anistiados, ex-combatente)
- assistencial        : amparos sociais (BPC/LOAS), auxílio-inclusão, pensões vitalícias especiais
- maternidade         : salario-maternidade
- reclusao            : auxilio-reclusao
- sem_especie_definida: codigos de despacho que vazaram para especie_cod na fonte
- outros              : demais espécies nao classificadas

In [0]:
# Codigos ja normalizados (sem zeros a esquerda), ver secao de normalizacao de codigos.
# "Afastamento" = beneficio por incapacidade temporaria (auxilio-doenca).
ESPECIES_AFASTAMENTO          = ["31", "91"]
# Auxilio-acidente e suplementar: indenizacao paga apos consolidacao de sequelas,
# nao e afastamento do trabalho.
ESPECIES_AUXILIO_ACIDENTE     = ["36", "94", "95"]
# 4 e 5: aposentadoria por invalidez de trabalhador rural (codigos legados)
ESPECIES_APOSENT_INVALIDEZ    = ["32", "92", "4", "5"]
# 7: aposentadoria por velhice de trabalhador rural (codigo legado)
ESPECIES_APOSENTADORIA        = ["38", "41", "42", "46", "51", "52", "57", "7"]
# 1: pensao por morte de trabalhador rural (codigo legado)
ESPECIES_PENSAO               = ["21", "22", "23", "56", "59", "84", "93", "1"]
ESPECIES_ASSISTENCIAL         = ["11", "18", "30", "85", "86", "87", "88", "96"]
ESPECIES_MATERNIDADE          = ["80"]
ESPECIES_RECLUSAO             = ["25"]
# Valores sem significado de espécie (lixo da fonte, ex.: CNPJ zerado no campo).
# Codigos de despacho que vazavam para especie_cod eram um efeito do desalinhamento
# de colunas na Bronze, corrigido em 01_bronze_ingestao.
ESPECIES_SEM_ESPECIE          = ["0", "00.000.000/0000-00"]

df_silver = df_silver.withColumn(
    "tipo_beneficio",
    when(col("especie_cod").isin(ESPECIES_AFASTAMENTO),        "afastamento")
    .when(col("especie_cod").isin(ESPECIES_AUXILIO_ACIDENTE),  "auxilio_acidente")
    .when(col("especie_cod").isin(ESPECIES_APOSENT_INVALIDEZ), "aposentadoria_invalidez")
    .when(col("especie_cod").isin(ESPECIES_APOSENTADORIA),     "aposentadoria")
    .when(col("especie_cod").isin(ESPECIES_PENSAO),        "pensao")
    .when(col("especie_cod").isin(ESPECIES_ASSISTENCIAL),  "assistencial")
    .when(col("especie_cod").isin(ESPECIES_MATERNIDADE),   "maternidade")
    .when(col("especie_cod").isin(ESPECIES_RECLUSAO),      "reclusao")
    .when(col("especie_cod").isin(ESPECIES_SEM_ESPECIE),   "sem_especie_definida")
    .otherwise("outros")
)

print("Classificacao de especie aplicada.")
display(
    df_silver
    .groupBy("tipo_beneficio")
    .count()
    .orderBy("count", ascending=False)
)


## 7. Validacao pre-gravacao

Verificacao das colunas derivadas e tipos aplicados.

In [0]:
print("=" * 50)
print("SCHEMA DA SILVER")
print("=" * 50)
df_silver.printSchema()

print("=" * 50)
print("DISTRIBUICAO cid_grupo")
print("=" * 50)
display(
    df_silver
    .groupBy("cid_grupo", "cid_status")
    .count()
    .orderBy("count", ascending=False)
)

print("=" * 50)
print("DISTRIBUICAO tipo_beneficio")
print("=" * 50)
display(
    df_silver
    .groupBy("tipo_beneficio")
    .count()
    .orderBy("count", ascending=False)
)


## 8. Gravacao da Silver

In [0]:
(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_DESTINO)
)

print(f"Tabela gravada: {TABELA_DESTINO}")


In [0]:
linhas_staging = df_staging.count()
linhas_silver  = spark.table(TABELA_DESTINO).count()
colunas_silver = len(spark.table(TABELA_DESTINO).columns)

print("=" * 50)
print("VALIDACAO STAGING vs SILVER")
print("=" * 50)
print(f"{'':30} {'STAGING':>8} {'SILVER':>8}")
print("-" * 50)
print(f"{'Linhas':30} {linhas_staging:>8,} {linhas_silver:>8,}")
print(f"{'Colunas Staging':30} {len(df_staging.columns):>8}")
print(f"{'Colunas Silver':30} {'':>8} {colunas_silver:>8}")
print("-" * 50)

if linhas_staging == linhas_silver:
    print("RESULTADO: OK - Nenhum registro perdido.")
else:
    diff = linhas_staging - linhas_silver
    print(f"RESULTADO: ATENCAO - Divergencia de {diff:,} registros.")

print("=" * 50)

display(spark.table(TABELA_DESTINO).limit(5))
